# 📧📫📧📪 <span style="color: white; background-color: steelblue;"><b> Envio de E-mails Curta suas Férias </b></span></p>

🧩  <span style="color: mediumseagreen;"><b> 1- Configuração Geral do Processo </b></span></p>
- Caixa compartilhada de envio gestaodepessoas@afpesp.org.br
- Arquivo Excel base: lista de colaboradores e e‑mails  
- Imagem embutida: enviada inline através de CID  
- Assunto: “Curta suas Férias !”  
    - Validação da existência dos arquivos necessários  
    - Criação da instância do Outlook
- Esse bloco garante que tudo esteja preparado antes do envio

📥 <span style="color: mediumseagreen;"><b> 2- Leitura e Validação da Base de E‑mails </b></span></p>
- O script lê o Excel informado
- Extraindo a coluna E-mail (obrigatório)  
- Valida cada endereço de e-mail
- Usa expressão regular (regex) para verificar estrutura básica:
    - E‑mails inválidos são ignorados  
    - E‑mails vazios são descartados
- Remove registros nulos ou duplicados
- Assegura integridade antes do disparo dos e‑mails

✍️ <span style="color: mediumseagreen;"><b> 3- Montagem do Corpo do E‑mail (HTML + Imagem Inline) </b></span></p>
- Para cada colaborador, o sistema constrói:
    - Saudação personalizada com o nome  
    - Texto em HTML estilizado  
    - Imagem inserida como inline attachment (Content-ID)  
    - Template limpo e padronizado
- Esse formato garante que o e‑mail:
    - Fique visualmente agradável
    - Não quebre em serviços populares de e-mail
    - Seja compatível com Outlook e webmail

📤 <span style="color: mediumseagreen;"><b> 4- Envio Automático pelo Outlook (via Shared Mailbox) </b></span></p>
- A automação utiliza:
    - win32com.client → automação COM do Outlook  
    - SentOnBehalfOfName → envio pela caixa compartilhada  
    - mail.Save() → salva rascunho automaticamente antes do envio  
    - mail.Send() → dispara a mensagem
- O processo garante:
    - Envio padronizado  
    - Compatibilidade com políticas de TI  
    - Controle via caixa compartilhada

📑 <span style="color: mediumseagreen;"><b> 5- Tratamento Individual de Cada Registro </b></span></p>
- Valida e-mail  
- Gera HTML  
- Anexa imagem  
- Define parâmetros de envio  
- Envia
- E registra:
    - Sucessos  
    - Falhas  
    - E‑mails inválidos

📊 <span style="color: mediumseagreen;"><b> 6- Relatório Final da Execução </b></span></p>
- Total de registros processados  
- Quantos e-mails foram enviados  
- Quantos falharam  
- Quantos e-mails foram descartados por invalidez
- Isso permite acompanhar rapidamente a eficiência da execução

In [ ]:
# Importando as Bibliotecas

import os
import pandas as pd
import re
import win32com.client as win32
from datetime import datetime, date
from openpyxl import load_workbook, Workbook
from pathlib import Path

# E-mail principal e caminhos de arquivos
CONTA_ENVIO = 'gestaodepessoas@afpesp.org.br'
EXCEL_PATH = r'X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.2 - Relatório de férias\2026\RELATÓRIO DE FERIAS 20260520 JUN 2026.xlsx'
IMAGE_PATH = r'X:\Gestão de Pessoas\Analytics\01 - Templates\01.0 - Imagens\art_ferias.jpg'

# Carregamento da Base de Controle de Processos
id = 25
path_registros_processos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'
registros_processos = pd.read_excel(path_registros_processos, sheet_name = "REGISTROS", engine='openpyxl')
wb_p = load_workbook(path_registros_processos)
ws_p = wb_p['REGISTROS']

# Controle de atualização de processo
tempo_0 = [id, datetime.today(), 0]
ws_p.append(tempo_0)
wb_p.save(path_registros_processos)

# Verifica se os arquivos necessários existem
def verificar_arquivos(EXCEL_PATH, IMAGE_PATH):
    if not os.path.exists(EXCEL_PATH):
        raise FileNotFoundError(f"Arquivo Excel não encontrado: {EXCEL_PATH}")
    if not os.path.exists(IMAGE_PATH):
        raise FileNotFoundError(f"Imagem não encontrada: {IMAGE_PATH}")

# Valida formato básico de e-mail
def email_valido(email):
    return bool(re.match(r'^[^@\s]+@[^@\s]+\.[^@\s]+$', str(email).strip()))

# Lê o arquivo Excel e retorna o DataFrame
def ler_excel(EXCEL_PATH):
    try:
        df = pd.read_excel(EXCEL_PATH, dtype=str)
        if 'E-mail' not in df.columns:
            raise ValueError("Coluna 'E-mail' não encontrada no Excel.")
        df = df.dropna(subset=['E-mail'])
        df['E-mail'] = df['E-mail'].astype(str).str.strip()
        if 'Nome' not in df.columns:
            df['Nome'] = 'Colaborador'
        else:
            df['Nome'] = df['Nome'].fillna('Colaborador').astype(str).str.strip()
        return df
    except Exception as e:
        print(f"Erro ao ler Excel: {str(e)}")
        return None

# Cria instância do Outlook
def criar_outlook():
    try:
        outlook = win32.Dispatch("Outlook.Application")
        return outlook
    except Exception as e:
        print(f"Erro ao conectar com Outlook: {str(e)}. Verifique se o Outlook está instalado.")
        return None

# Monta o corpo HTML do e-mail
def montar_html(nome):
    return f"""
    <html>
    <body style="font-family: Arial, sans-serif;">
        <h2 style="color: #0070C0;">Olá, {nome}!</h2>
        <img src="cid:image_cid" alt="Arte de Férias" style="max-width: 100%; height: auto;">
        <p>Desejamos ótimas férias!</p>
        <br>
        <p>Atenciosamente,<br>Equipe Gestão de Pessoas</p>
    </body>
    </html>
    """

# Envia e-mail com imagem embutida
def enviar_email(outlook, destinatario, nome, assunto, image_path):
    try:
        mail = outlook.CreateItem(0)  # 0 = olMailItem
        mail.To = destinatario
        mail.Subject = assunto

        # Configura remetente como caixa compartilhada (Shared Mailbox)
        # Shared Mailboxes não aparecem em Session.Accounts — usar CreateRecipient
        remetente = outlook.Session.CreateRecipient(CONTA_ENVIO)
        remetente.Resolve()
        if remetente.Resolved:
            mail.Sender = remetente.AddressEntry
            mail.SentOnBehalfOfName = CONTA_ENVIO
        else:
            print(f"ERRO: Não foi possível resolver '{CONTA_ENVIO}'. Verifique se você tem permissão de 'Enviar como' nessa caixa compartilhada.")
            return False

        # Anexa a imagem como inline (CID)
        attachment = mail.Attachments.Add(os.path.abspath(image_path))
        attachment.PropertyAccessor.SetProperty(
            "http://schemas.microsoft.com/mapi/proptag/0x3712001E",
            "image_cid"
        )

        mail.HTMLBody = montar_html(nome)
        mail.Save()   # Salva rascunho antes de enviar (evita perda em falhas)
        mail.Send()
        print(f"E-mail enviado com sucesso para {destinatario} ({nome})")
        return True

    except Exception as e:
        print(f"Erro ao enviar e-mail para {destinatario}: {str(e)}")
        return False

def main():

    try:
        verificar_arquivos(EXCEL_PATH, IMAGE_PATH)

        df = ler_excel(EXCEL_PATH)
        if df is None or df.empty:
            print("Nenhum dado válido encontrado no Excel.")
            return

        print(f"Lidos {len(df)} registros do Excel.")

        outlook = criar_outlook()
        if outlook is None:
            return

        assunto = "Curta suas Férias!"

        sucessos = 0
        falhas = 0
        invalidos = 0

        for _, row in df.iterrows():
            email = row['E-mail']
            nome = row['Nome']

            if not email_valido(email):
                print(f"E-mail inválido pulado: {email}")
                invalidos += 1
                continue

            if enviar_email(outlook, email, nome, assunto, IMAGE_PATH):
                sucessos += 1
            else:
                falhas += 1

        print("\n" + "=" * 50)
        print("RELATÓRIO DE EXECUÇÃO:")
        print(f"  Enviados com sucesso : {sucessos}")
        print(f"  Falhas de envio      : {falhas}")
        print(f"  E-mails inválidos    : {invalidos}")
        print(f"  Total processados    : {sucessos + falhas + invalidos}")
        print("=" * 50)

    except Exception as e:
        print(f"Erro geral no script: {str(e)}")

if __name__ == "__main__":
    main()

tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print('----------------------------------------------------------------------------------------------------')

Lidos 109 registros do Excel.
E-mail enviado com sucesso para barbosa-laiane@hotmail.com (VERA LUCIA BARBOSA MOTA)
E-mail enviado com sucesso para wagneluz@afpesp.org.br (VAGNER NOVAIS LUZ)
E-mail enviado com sucesso para rsantos@afpesp.org.br (ROGERIO DUARTE DOS SANTOS)
E-mail enviado com sucesso para tmorais@afpesp.org.br (TIAGO FARIA MORAIS)
E-mail enviado com sucesso para rumsilva@afpesp.org.br (RUTE MARIA DA SILVA)
E-mail enviado com sucesso para elmacedo@afpesp.org.br (ELENICE GUILHERMINO DE MACEDO)
E-mail enviado com sucesso para jamachad@afpesp.org.br (JAILTON ALBERTO MACHADO)
E-mail enviado com sucesso para antmauro2022@gmail.com (ANTONIO MAURO BENEDITO)
E-mail enviado com sucesso para jesantos@afpesp.org.br (JESUABE QUIRINO DOS SANTOS)
E-mail enviado com sucesso para ne9331952@gmail.com (CARLOS AUGUSTO BARBOSA DA SILVA)
E-mail enviado com sucesso para gesantos@afpesp.org.br (GENILSON APARECIDO DOS SANTOS)
E-mail enviado com sucesso para atoop724@gmail.com (ANTONIO DE LUCENA M

TypeError: unsupported operand type(s) for -: 'datetime.datetime' and 'int'